In [ ]:
!pip install -q faster-whisper groq gtts gradio typer==0.24.0

import gradio as gr
from faster_whisper import WhisperModel
from groq import Groq
from gtts import gTTS
import os
import uuid

# =========================
# SECRET KEY (Colab Secrets)
# =========================
from google.colab import userdata
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

client = Groq()

# =========================
# MODEL
# =========================
model = WhisperModel("base", compute_type="float16")

# =========================
# STT
# =========================
def speech_to_text(audio_path):
    if not audio_path:
        return ""

    segments, _ = model.transcribe(audio_path)

    text = ""
    for segment in segments:
        text += segment.text + " "

    return text.strip()

# =========================
# LLM
# =========================
def get_llm_response(text):
    res = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": text}
        ]
    )
    return res.choices[0].message.content

# =========================
# TTS
# =========================
def text_to_speech(text):
    file = f"audio_{uuid.uuid4().hex}.mp3"
    gTTS(text=text, lang="en").save(file)
    return file

# =========================
# MAIN FUNCTION (FIXED)
# =========================
def voice_assistant(audio):

    # FIX 1: avoid first click issue
    if audio is None or audio == "":
        return None, "No audio detected", ""

    # STT
    user_text = speech_to_text(audio)
    if not user_text:
        return None, "Could not understand audio", ""

    # LLM
    ai_text = get_llm_response(user_text)

    # TTS
    audio_out = text_to_speech(ai_text)

    # FIX 2: prevent audio replay loop
    return gr.update(value=audio_out, autoplay=True), ai_text, user_text

# =========================
# MODERN UI
# =========================
css = """
body {
    background: linear-gradient(135deg, #0f172a, #020617);
}

.container {
    max-width: 900px;
    margin: auto;
    padding: 30px;
}

.card {
    background: rgba(255,255,255,0.06);
    border: 1px solid rgba(255,255,255,0.12);
    border-radius: 24px;
    padding: 25px;
    backdrop-filter: blur(14px);
    box-shadow: 0 10px 40px rgba(0,0,0,0.4);
}

.title {
    font-size: 36px;
    font-weight: bold;
    text-align: center;
    color: white;
}

.subtitle {
    text-align: center;
    color: #94a3b8;
    margin-bottom: 20px;
}

textarea {
    border-radius: 14px !important;
}
"""

with gr.Blocks(css=css) as demo:

    with gr.Column(elem_classes="container"):

        with gr.Column(elem_classes="card"):

            gr.HTML("""
            <div class="title">🎙️ AI Voice Assistant</div>
            <div class="subtitle">Speak → AI Responds → Voice Output</div>
            """)

            audio_input = gr.Audio(
                sources=["microphone"],
                type="filepath",
                label="Speak here"
            )

            user_text = gr.Textbox(
                label="Your Speech",
                lines=2
            )

            ai_text = gr.Textbox(
                label="AI Response",
                lines=6
            )

            audio_output = gr.Audio(
                label="Voice Response",
                autoplay=True
            )

    # AUTO TRIGGER (NO BUTTON)
    audio_input.change(
        fn=voice_assistant,
        inputs=audio_input,
        outputs=[audio_output, ai_text, user_text]
    )

demo.launch(debug=True)